# Voice Clone Detection — Tier-1 CNN GPU Training (FULL 12k files)

**Setup first:** `Runtime > Change runtime type > T4 GPU` (else training runs on CPU and takes hours).

You need 2 files from your laptop in this same folder:
- `colab_train_tier1.ipynb` (this notebook — `File > Upload notebook` in Colab)
- `sih-code.zip` (your CURRENT code bundle — upload it when Cell 3 asks)

Flow: GPU check → upload code → install → download MLAAD FULL (~12.5k wavs) → smoke test → full train (20 epochs) → download checkpoint.

In [ ]:
# Cell 1 — GPU check. Must show a T4 + torch cuda=True. If not, fix Runtime first.
!nvidia-smi
!python --version
import torch
print('torch:', torch.__version__)
print('cuda_available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU — set Runtime > Change runtime type > T4 GPU, then re-run')

## Upload your current code
Run the next cell — it will prompt a file picker. Select **`sih-code.zip`** from your laptop. That zip was built from your working tree (`src/`, `scripts/`, `pyproject.toml`), so it contains the NEW 597k-param `tier1_cnn.py`, not the stale git HEAD.

In [ ]:
# Cell 2 — upload + unpack code bundle
from google.colab import files
import shutil, pathlib

print('Select sih-code.zip in the picker...')
up = files.upload()  # choose sih-code.zip
assert 'sih-code.zip' in up, f'Expected sih-code.zip, got {list(up.keys())}'

shutil.rmtree('/content/sih', ignore_errors=True)
pathlib.Path('/content/sih').mkdir(parents=True, exist_ok=True)
shutil.unpack_archive('/content/sih-code.zip', '/content/sih')
!ls -R /content/sih | head -40
!ls /content/sih/src/voice_detection/

In [ ]:
# Cell 3 — deps. Colab already ships CUDA torch; do NOT reinstall torch (slow). Just add helpers.
!pip -q install huggingface_hub hf_transfer numpy
!python -c "import torch; print(torch.__version__, torch.cuda.is_available())"
%cd /content/sih
!pwd; ls

In [ ]:
# Cell 4 — download MLAAD FULL (~6070 genuine + ~6400 spoof + ~752 unseen). Takes ~10-20 min.
# Uses scripts/download_mlaad_tiny.py --full (same script that built your local data/).
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
!python scripts/download_mlaad_tiny.py --full
!echo '--- counts ---'; echo -n 'genuine: '; ls data/genuine | wc -l; echo -n 'spoof: '; ls data/spoof | wc -l; echo -n 'unseen genuine: '; ls data_unseen/genuine | wc -l; echo -n 'unseen spoof: '; ls data_unseen/spoof | wc -l
!du -sh data data_unseen

In [ ]:
# Cell 5 — CUDA smoke test (~2 min). Proves training loop works on GPU before the long run.
!PYTHONPATH=src python -m voice_detection.train_tier1 --synthetic 800 --epochs 2 --batch-size 128 --num-workers 2 --out /tmp/smoke.pt

In [ ]:
# Cell 6 — FULL TRAIN (the real run, T4 ~30-60 min for 20 epochs on ~12.5k clips).
# --num-workers 2 is deliberate: Colab default 4 can hang. Bump --batch-size to 256 only if VRAM allows.
!PYTHONPATH=src python -m voice_detection.train_tier1 \
  --genuine-dir data/genuine --spoof-dir data/spoof \
  --epochs 20 --lr 0.001 --batch-size 128 --crops 2 \
  --num-workers 2 --out checkpoints/tier1_mlaad.pt

In [ ]:
# Cell 7 — verify checkpoint matches CURRENT arch (597k params) + download it.
!ls -lh checkpoints/
!PYTHONPATH=src python -c "import torch,sys; sys.path.insert(0,'src'); from voice_detection.tier1_cnn import Tier1CausalCNN; m=Tier1CausalCNN(); ckpt=torch.load('checkpoints/tier1_mlaad.pt',map_location='cpu'); m.load_state_dict(ckpt['model_state_dict']); print('LOAD OK, params:', m.parameter_count); print('config:', ckpt.get('config'))"
from google.colab import files
files.download('checkpoints/tier1_mlaad.pt')
# Optional Drive backup (uncomment):
# from google.colab import drive; drive.mount('/content/drive')
# !cp checkpoints/tier1_mlaad.pt "/content/drive/MyDrive/tier1_mlaad.pt"; print('backed up to Drive')

## Back on your laptop

1. Copy the downloaded `tier1_mlaad.pt` → `C:\Users\theka\Downloads\sih\checkpoints\tier1_mlaad.pt` (overwrite the stale 125k-param file).
2. Verify:
```powershell
$env:PYTHONPATH = "src"
python -c "import torch; from voice_detection.tier1_cnn import Tier1CausalCNN; m=Tier1CausalCNN(); m.load_state_dict(torch.load('checkpoints/tier1_mlaad.pt',map_location='cpu')['model_state_dict']); print('OK', m.parameter_count)"
```
3. Run server with it:
```powershell
$env:TIER1_CHECKPOINT = "checkpoints\tier1_mlaad.pt"
uvicorn voice_detection.api:app --app-dir src --reload
```
4. Copy the final `val_acc / val_eer / sliding-window` lines from Cell 6 into `README.md` (old 0.72/0.27 numbers are from the previous arch).